# TP 2 — Partie 1 : ingérer du texte libre et du JSON en couche Bronze

**Objectif :** faire entrer dans le Lakehouse deux sources qui ne sont pas des tables bien rangées :

| Source | Format | Ce qui la rend difficile |
|---|---|---|
| Avis des clients | CSV, avec un commentaire en texte libre | le texte contient des retours à la ligne, des guillemets, des accents |
| Suivi des colis par le transporteur | JSON, un message par colis | structure imbriquée (listes d'événements), format qui évolue avec le temps |

La règle de la couche Bronze ne change pas : **on garde la donnée telle qu'elle est arrivée**, avec deux colonnes techniques (`_source`, `_ingere_le`).

**Prérequis :** le TP 1 est terminé (les tables Silver `orders`, `customers`… existent).

> **⚠️ Un seul notebook actif à la fois.** Avant de commencer, arrêtez le noyau des autres notebooks : panneau *Running Terminals and Kernels* > *Shut Down All*.

In [ ]:
import os
import datetime
import boto3
from pyspark.sql import SparkSession, functions as F

spark = SparkSession.builder.appName("tp2-bronze").getOrCreate()
s3 = boto3.client("s3", endpoint_url=os.environ["S3_ENDPOINT"])
DATE_INGESTION = datetime.date.today().isoformat()


def a_completer(quoi):
    """Signale clairement une partie du TP qui n'a pas encore été écrite."""
    raise NotImplementedError(f"⚠️ Cellule à compléter : {quoi}")


spark.sql("SHOW NAMESPACES IN lakehouse").show()

## Étape 1 — Déposer les sources dans la zone d'atterrissage

Les deux sources arrivent dans `s3://landing/` :
- le fichier des avis, copié tel quel ;
- les messages du transporteur, un fichier JSON par mois. Ils sont produits par le script `generer_suivi.py`, qui les reconstruit à partir des vraies dates de livraison des commandes Olist (les transporteurs, les lieux de transit et les incidents sont simulés).

In [ ]:
cle = f"olist/avis/date_ingestion={DATE_INGESTION}/olist_order_reviews_dataset.csv"
s3.upload_file("/home/etudiant/work/data/olist/olist_order_reviews_dataset.csv", "landing", cle)
print("déposé :", f"s3://landing/{cle}")

In [ ]:
!python /opt/lakehouse/scripts/generer_suivi.py

## Étape 2 — Le piège du texte libre

Un fichier CSV range normalement **un enregistrement par ligne**. Mais un client peut écrire un commentaire sur plusieurs lignes : le retour à la ligne est alors *à l'intérieur* d'un champ entre guillemets.

Comparez le nombre de lignes du fichier, une lecture CSV naïve et le nombre réel d'avis.

In [ ]:
CHEMIN_AVIS = "s3a://landing/olist/avis/"

lignes_fichier = spark.read.text(CHEMIN_AVIS).count()
naif = spark.read.option("header", True).csv(CHEMIN_AVIS)

print("lignes physiques du fichier :", lignes_fichier)
print("lecture CSV naïve           :", naif.count(), "enregistrements")
naif.filter(F.col("review_score").isNull() | ~F.col("review_score").rlike("^[1-5]$")) \
    .select("review_id", "review_score", "review_comment_message").show(5, truncate=40)

> **❓ Question 1.** Pourquoi la lecture naïve trouve-t-elle plus d'enregistrements qu'il n'y a d'avis ? Que contiennent les lignes affichées ? Qu'arriverait-il aux analyses si on chargeait ces données en l'état ?

## Étape 3 — Créer `bronze.avis`

Complétez la fonction `bronze_avis()`. Elle doit :
1. lire **en texte** (sans `inferSchema`) les fichiers de `CHEMIN_AVIS`, avec la ligne d'en-tête ;
2. autoriser les enregistrements sur plusieurs lignes : options `multiLine` à `True` et `escape` à `'"'` (un guillemet dans un commentaire est doublé) ;
3. ajouter `_source` (`F.col("_metadata.file_path")`) et `_ingere_le` (`F.current_timestamp()`) ;
4. écrire la table Iceberg `lakehouse.bronze.avis` en la remplaçant si elle existe.

In [ ]:
def bronze_avis():
    df = a_completer("lecture du CSV multi-lignes")  # À COMPLÉTER

    # À COMPLÉTER : ajout de _source et _ingere_le, puis écriture dans lakehouse.bronze.avis
    return spark.table("lakehouse.bronze.avis").count()


print("bronze.avis :", bronze_avis(), "lignes")
spark.table("lakehouse.bronze.avis").groupBy("review_score").count().orderBy("review_score").show()

> **❓ Question 2.** Vérifiez que `review_score` ne contient plus que les valeurs 1 à 5. Combien d'avis ont un commentaire (`review_comment_message` non vide) ?

In [ ]:
# À COMPLÉTER : nombre d'avis avec un commentaire

## Étape 4 — Regarder un message JSON

Un message du transporteur décrit un colis. Affichez-en un : repérez les **objets imbriqués** (`transporteur`, `lieu`) et les **listes** (`evenements`).

In [ ]:
CHEMIN_SUIVI = "s3a://landing/olist/suivi/"

brut = spark.read.text(CHEMIN_SUIVI)
print(brut.first()["value"])

Laissez Spark deviner la structure de tous les messages :

In [ ]:
devine = spark.read.json(CHEMIN_SUIVI)
devine.printSchema()
devine.groupBy("version_schema").count().show()

> **❓ Question 3.** Le format des messages a changé en janvier 2018 (`version_schema = 2`). Quels champs sont apparus ? Que vaut `incidents` dans un message de 2017 ? À quoi correspond la colonne `_corrupt_record`, si elle apparaît ?

## Étape 5 — Créer `bronze.suivi_brut`

En Bronze, on **ne découpe pas** le JSON : on garde chaque message comme un texte, dans une colonne `value`. Ainsi, aucun message n'est perdu, même tronqué, et un changement de format ne casse pas l'ingestion. Le découpage se fera en Silver.

Créez `lakehouse.bronze.suivi_brut` à partir de `spark.read.text(CHEMIN_SUIVI)`, avec les colonnes `_source` et `_ingere_le`. La colonne `mois`, déduite du nom des dossiers (`mois=2018-03`), est conservée.

In [ ]:
suivi = a_completer("lecture des messages en texte + colonnes techniques")  # À COMPLÉTER
suivi.writeTo("lakehouse.bronze.suivi_brut").using("iceberg").createOrReplace()

print("bronze.suivi_brut :", spark.table("lakehouse.bronze.suivi_brut").count(), "messages")
spark.table("lakehouse.bronze.suivi_brut").printSchema()

> **❓ Question 4.** Pourquoi garder le JSON brut en Bronze plutôt que la version découpée par `spark.read.json` ? Donnez deux situations où cela vous sauve.

## Bilan

In [ ]:
spark.sql("SHOW TABLES IN lakehouse.bronze").show()
for t in ["avis", "suivi_brut"]:
    print(f"bronze.{t:12s}", spark.table(f"lakehouse.bronze.{t}").count(), "lignes")